# Chapter 27 – Worked examples

This notebook runs the examples of [Chapter 27](../index.md) live. The code lives in the modules next to it
(`pareto.py`, `dominator_filter.py`, `repertoire_cells.py`, `augmecon.py`, `ch27_ieee30.py`); the cells only call
them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import augmecon
import ch27_ieee30
import dominator_filter
import numpy as np
import pandas as pd
import pareto
import repertoire_cells

pd.set_option('display.width', 200)

## Supported and unsupported points

The five points of Exercise 1 are all non-dominated. A weighted sum $w f_1 + (1 - w) f_2$ reaches only the
points on the convex hull of the front; the point $(4, 3.5)$ lies in a non-convex dent and no weight selects it.
The $\varepsilon$-constraint method and an achievement scalarizing function do reach it.

In [ ]:
points = pareto.POINTS
print('non-dominated :', points[pareto.non_dominated_mask(points)].tolist())
print('supported     :', points[pareto.supported_mask(points)].tolist())
print('winners of 1001 weights:', sorted(pareto.weighted_sum_sweep(points)))
print('eps-constraint f2 <= 3.6 :', points[pareto.epsilon_constraint(points, 0, np.array([0, 3.6]))])
print('ASF, reference (4, 3)    :', points[pareto.achievement_scalarizing(points, np.array([4.0, 3.0]))])

## Non-dominated sorting, crowding distance and indicators

`fast_non_dominated_sort` ranks a cloud of random points, `crowding_distance` spreads the first front, and
`hypervolume_2d` adds rectangles between the sorted front and the reference point.

In [ ]:
rng = np.random.default_rng(seed=27)
cloud = rng.random((1000, 3))
ranks = pareto.fast_non_dominated_sort(cloud)
print(f'{ranks.max() + 1} fronts, {np.sum(ranks == 0)} points in the first')
for factor in (1.0, 1.1, 2.0):
    hv = pareto.hypervolume_2d(points, factor * points.max(axis=0))
    print(f'hypervolume of the Exercise 1 points, reference = {factor} x nadir: {hv:.2f}')

## ToOp's dominator filter is not a Pareto filter

`dominator_mask` is a port of `get_dominator_mask` (`optimizer/ac/select_strategy.py`). Counterexample (a): a
non-dominated row is removed; (b): dominated rows are kept. `pareto_mask` is the true non-dominated set with the
fitness maximized and every metric minimized.

In [ ]:
for name, df in (('(a)', dominator_filter.counterexample_a()), ('(b)', dominator_filter.counterexample_b())):
    mask = dominator_filter.dominator_mask(df, ['split_subs'], dominator_filter.OBSERVED)
    print(name)
    print(df.assign(dominator=mask, pareto=dominator_filter.pareto_mask(df, dominator_filter.OBSERVED)))

Swapping the order of the target metrics changes the mask.

In [ ]:
df = dominator_filter.order_example()
first = dominator_filter.dominator_mask(df, ['split_subs', 'switching_distance'], ['disconnections'])
second = dominator_filter.dominator_mask(df, ['switching_distance', 'split_subs'], ['disconnections'])
print(df.assign(split_subs_first=first, switching_distance_first=second))

## Repertoire cells are clipped

With `n_cells_per_dim = (5, 45)` a descriptor outside the grid lands in the last cell of its axis.

In [ ]:
for d in ((1, 2), (4, 44), (4, 60), (7, 44)):
    print(d, '-> cell', repertoire_cells.cell_index(d), ', exact cell', repertoire_cells.exact_cell_index(d))
table = repertoire_cells.synthetic_repertoire()
result = repertoire_cells.analyse(table)
edge = len(result['edge_cell_elites'])
print(f'{len(table)} elites, {result["dominated_count"]} dominated, {edge} in clipped edge cells')

## An exact bi-objective front

AUGMECON2 on the facility location instance of Chapter 23 (opening cost against assignment cost), and the exact
front of IEEE 30 line switching with at most two switched lines (overload energy against switched lines).

In [ ]:
front, solved = augmecon.augmecon2()
print(f'facility location: {front} with {solved} MILPs')
case = ch27_ieee30.load_case()
exact, sets, count = ch27_ieee30.enumerate_front(case, k_max=2)
for (e, k), lines in zip(exact, sets, strict=True):
    print(f'{int(k)} switched line(s) {list(lines)}: {e:.1f} MW of N-1 overload')
print(f'{count} evaluations')